# Prolog Program Explainer using LangChain

This notebook creates a tool that provides natural language explanations of Prolog programs using LangChain.

In [1]:
# Import necessary libraries
from langchain.prompts import PromptTemplate
from pydantic import BaseModel, Field
from typing import List
from langchain.output_parsers import PydanticOutputParser
from langchain.schema import StrOutputParser
from langchain_ollama import OllamaLLM
import shutil
import re
import json

In [2]:
ollama_exists = shutil.which('ollama') is not None

if not ollama_exists:
    # Install Ollama (if not already installed)
    !curl -fsSL https://ollama.com/install.sh | sh

    # Start the Ollama server
    !ollama serve

In [3]:
# Define the output structure with a Pydantic model
class PrologExplanation(BaseModel):
    summary: str = Field(description="A brief summary of what the Prolog program does")
    primitive: List[dict] = Field(description="List of dependent primitives and their descriptions")
    explanation: str = Field(description="How the program logic works")
    
def explain_prolog(llm, prompt_template, prolog_code, variables=['prolog_code']):
    """Generate a natural language explanation of a Prolog program.
    
    Args:
        llm: The language model to use
        prompt: The prompt template to use
        prolog_code (str): The Prolog code to explain
        
    Returns:
        PrologExplanation: Structured explanation of the Prolog program
    """
    # Initialize the output parser
    parser = PydanticOutputParser(pydantic_object=PrologExplanation)
    
    prompt = PromptTemplate(
        template=prompt_template,
        input_variables=variables,
        partial_variables={"JSON_format": parser.get_format_instructions()}
    )
    
    # Get raw string response from the LLM
    chain = prompt | llm | StrOutputParser()
    raw_result = chain.invoke({"prolog_code": prolog_code})
    
    print(f"===== Generated Raw Text =====\n{raw_result}")
    
    # Parse the raw string into a structured PrologExplanation object
    try:
        # Try to parse the output directly
        parsed_result = parser.parse(raw_result)
        
        # print(f"\n===== Parsed Prolog Explanation =====\n{parsed_result}")
        print(f"\n===== Parsed Text =====")

        print(f"Summary: {parsed_result.summary}\n")
        print("Primitives:")
        for pred in parsed_result.primitive:
            print(f"- {pred.get('name')}: {pred.get('description')}")
        print(f"\nExplanation: {parsed_result.explanation}\n")
        return parsed_result
    except Exception as e:
        # If direct parsing fails, attempt to extract structured data from text
        print(f"Parsing error: {e}")
        print("Returning raw output instead. Check format of LLM response.")
        return raw_result

In [4]:
# Llama 3.2 3B
llm = OllamaLLM(
    model = 'llama3.2'
)

In [5]:
explanation_template = """
You are an expert Prolog interpreter who explains what the Prolog code does in English to non-programmers.

I. Analyze the following Prolog program which is divided into three sections:

1. Target rules: The main predicates that define the goal of the program
2. Primitives: Supporting predicates that help implement the target rules
3. Biases: Input and output signature directions for the primitives

{prolog_code}

II. Your response MUST follow the below JSON format EXACTLY with no additional text:

{JSON_format}

YOUR EXPLANATION MUST:

In "summary": Provide a 1-2 sentence overview of what the entire Prolog program accomplishes.
In "primitive": List each primitive from Target rules and explain what they do as dictionaries with "name" and "description" keys.
In "explanation": Explain what the Target rules do to a person without any programming or technical experience.

IMPORTANT FORMATTING RULES:

Return ONLY valid parseable JSON with the exact field names shown above.
Ensure all strings are properly escaped for valid JSON.
DO NOT return JSON schema information or "properties" fields.
Do not include markdown formatting symbols, code blocks, or explanatory text outside the JSON structure.
"""

## Example Usage

In [6]:
# Example Prolog code - Family relationships
family_prolog = """
%%%%%%%%%%%%%%%%%%%% Target rules %%%%%%%%%%%%%%%%%%%%
grandparent(X, Z) :- parent(X, Y), parent(Y, Z).

%%%%%%%%%%%%%%%%%%%% Primitives %%%%%%%%%%%%%%%%%%%%
parent(X, Y) :- father(X, Y).
parent(X, Y) :- mother(X, Y).

%%%%%%%%%%%%%%%%%%%% Biases %%%%%%%%%%%%%%%%%%%%
# direction(grandparent,(in,out)).
direction(parent,(in,out)). 
direction(father,(in,out)). 
direction(mother,(in,out)). 
"""

explanation = explain_prolog(llm, explanation_template, family_prolog)

===== Generated Raw Text =====
{"summary": "This Prolog program finds the grandparent of a given person by recursively tracing their ancestors until it reaches an individual who is at least one generation away from them.", "primitive": [{"name": "parent", "description": "This primitive checks if a given individual has a parent, either through the male (father) or female (mother) lineage."}, {"name": "grandparent", "description": "This primitive finds the grandparent of an individual by linking two parent relationships together."}, {"name": "father", "description": "This primitive determines if one individual is the father of another."}, {"name": "mother", "description": "This primitive determines if one individual is the mother of another."}], "explanation": "The program starts with a target rule that says if you know an individual's grandparent, then their parent and grandparent are related. It uses two special primitives called parent and father/mother to figure this out. The parent 

## Custom Prolog Code Explanation

Without code comments

In [23]:
# Example Prolog code - List processing
list_prolog = """

%%%%%%%%%%%%%%%%%%%% Target rules %%%%%%%%%%%%%%%%%%%%

inv_ho_0(A,B):- same_circuit(A,B),linear_path(B,A).
inv_ho_1(A,B):- same_circuit(A,B),not(linear_path(B,A)).

partition(A,B,C):- all(inv_ho_0,A,B),all(inv_ho_1,A,C).

%%%%%%%%%%%%%%%%%%%% Primitives %%%%%%%%%%%%%%%%%%%%
all(P, A, L):- 
    findall(H, call(P, A, H), L).

linearpath(A,B):- same_gate(A,B).
linearpath(A,B):- not(branches(A)),is_connected(A,C),linearpath(C,B).
not_linearpath(A,B) :- not(linearpath(A,B)).

same_circuit(A, B) :- gate(A), gate(B), N is A // 100, M is B // 100, N == M.

%%%%%%%%%%%%%%%%%%%% Biases %%%%%%%%%%%%%%%%%%%%
direction(partition,(in,out,out)). 
direction(linear_path,(in,out)).
direction(not_linear_path,(in,out)).
direction(same_circuit,(in,out)).
direction(all,((in,out),in,out)).

"""

explanation = explain_prolog(llm, explanation_template, list_prolog)

===== Generated Raw Text =====
{"summary": "This Prolog program determines whether a circuit is connected and if it has linear paths within that circuit.", "primitive": [{"name": "same_circuit", "description": "Checks if two gates have the same gate ID with different node IDs, indicating they are in the same circuit."}, {"name": "linearpath", "description": "Verifies if a path between two nodes is linear (i.e., only one gate) or not linear."}, {"name": "branches", "description": "Checks for non-linear gate configurations within the circuit."}, {"name": "is_connected", "description": "Determines whether multiple gates are connected in series."}, {"name": "all", "description": "Recursively gathers results from a query to an operation."}], "explanation": "The program has three main target rules: inv_ho_0, inv_ho_1, and partition. These rules determine if a circuit is linear or not and then checks the entire circuit for connectivity. The primitives used in these rules help with operations 

With code comments

In [ ]:
# Example Prolog code - List processing
list_prolog = """

%%%%%%%%%%%%%%%%%%%% Target rules %%%%%%%%%%%%%%%%%%%%

% Gates affected by a test and those not affected
inv_ho_0(A,B):- same_circuit(A,B),linearpath(B,A).
inv_ho_1(A,B):- same_circuit(A,B),not(linearpath(B,A)).

% Given Gate A find gates on the same linear path as A and gates on other paths
partition(A,B,C):- all(inv_ho_0,A,B),all(inv_ho_1,A,C).

%%%%%%%%%%%%%%%%%%%% Primitives %%%%%%%%%%%%%%%%%%%%

% Higher Order primitive
all(P, A, L):- 
    findall(H, call(P, A, H), L).

% Is gate A to gate B a path without branches?
linearpath(A,B):- same_gate(A,B).
linearpath(A,B):- not(branches(A)),is_connected(A,C),linearpath(C,B).
not_linearpath(A,B) :- not(linearpath(A,B)).

% Does gate A share a linear path with gate B (B precedes A if A \\== B)?
same_circuit(A, B) :- gate(A), gate(B), N is A // 100, M is B // 100, N == M.

%%%%%%%%%%%%%%%%%%%% Biases %%%%%%%%%%%%%%%%%%%%

direction(partition,(in,out,out)). 
direction(linear_path,(in,out)).
direction(not_linear_path,(in,out)).
direction(same_circuit,(in,out)).
direction(all,((in,out),in,out)).

"""

explanation = explain_prolog(llm, explanation_template, list_prolog)

===== Generated Raw Text =====
{"summary": "The Prolog program analyzes gate connections and determines which gates share linear paths.", 
"primitive": [{"name": "all", "description": "A higher-order primitive to find all solutions for a predicate"}, {"name": "linearpath", "description": "Checks if two gates are on the same linear path without branches"}, {"name": "same_circuit", "description": "Determine if two gates share the same circuit but not necessarily the same linear path"}, {"name": "not_linearpath", "description": "Checks if a path is not linear"}, {"name": "branches", "description": "Checks if there are branches between gates"}], 
"explanation": "The program starts by identifying gates that are affected by a test and those that are not. Then, it finds the gates on the same linear path as a given gate and those on other paths. The \"partition\" rule groups gates into two categories based on their connection status."}

===== Parsed Text =====
Summary: The Prolog program analy